# DataCleanEnv — GRPO Training on Kaggle

Train an LLM to clean messy CSV data using GRPO (Group Relative Policy Optimization).

## Kaggle Setup
1. **Enable GPU**: Settings → Accelerator → GPU P100
2. **Enable Internet**: Settings → Internet → On
3. Run all cells in order

## Hardware
- **P100 (16GB VRAM)**: Can train Qwen2.5-3B with 4-bit quantization
- **2x T4 (if available)**: Can train larger models

## 1. Clone Repository & Install Dependencies

In [ ]:
# Clone the repository
!git clone https://github.com/YOUR_USERNAME/openenv-datacleaning-agent.git
%cd openenv-datacleaning-agent

# Install dependencies
!pip install -q openenv-core[core] transformers accelerate bitsandbytes peft trl datasets
!pip install -q fastapi fastmcp uvicorn pandas numpy
!pip install -e ./data_clean_env

print("✅ Dependencies installed")

## 2. GPU Check & Configuration

In [ ]:
import torch
import os

# Check GPU
print("=" * 50)
print("GPU STATUS")
print("=" * 50)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name}")
    print(f"✅ VRAM: {gpu_memory:.1f} GB")
else:
    print("❌ No GPU! Enable GPU in Settings → Accelerator")
    raise RuntimeError("GPU required")

# Select model based on GPU memory
if gpu_memory >= 16:  # P100
    MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
else:
    MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"\n📦 Selected model: {MODEL_ID}")

# ============================================
# 🔑 HUGGINGFACE TOKEN
# ============================================
# Get token from: https://huggingface.co/settings/tokens
# Or add as Kaggle Secret: Add-ons → Secrets → HF_TOKEN

HF_TOKEN = ""  # <-- Paste your token here, or use Kaggle Secrets

# Try Kaggle secrets first
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
        print("✅ HF_TOKEN loaded from Kaggle Secrets")
    except:
        print("⚠️ No HF_TOKEN - downloads may be slower")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ Logged in to HuggingFace")

# Training config - Optimized for P100
CONFIG = {
    "learning_rate": 5e-5,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "max_completion_length": 384,
    "num_generations": 2,
    "beta": 0.1,
    "temperature": 0.7,
    "output_dir": "./dataclean-grpo-kaggle",
    "logging_steps": 1,
    "save_steps": 25,
}

print(f"\n⏱️ Estimated training time: ~45-60 min on P100")

## 3. Generate Training Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import random
import string

tasks_dir = Path("data_clean_env/tasks")
tasks_dir.mkdir(parents=True, exist_ok=True)

def add_noise(value, noise_type="mixed"):
    """Add realistic noise to data."""
    if pd.isna(value):
        return value
    if random.random() < 0.1:
        return np.nan
    if isinstance(value, str):
        if random.random() < 0.2:
            value = "  " + value + "  "
        if random.random() < 0.15:
            value = value.upper() if random.random() < 0.5 else value.lower()
    return value

# Generate diverse datasets
DATASETS = {
    "customers": {
        "columns": ["name", "email", "phone", "city", "signup_date"],
        "generator": lambda i: {
            "name": random.choice(["John Smith", "Jane Doe", "Bob Wilson", "Alice Brown", "Charlie Davis"]),
            "email": f"user{i}@{'gmail.com' if random.random() > 0.3 else 'yahoo.com'}",
            "phone": f"{random.randint(100,999)}-{random.randint(100,999)}-{random.randint(1000,9999)}",
            "city": random.choice(["New York", "Los Angeles", "Chicago", "Houston", "Phoenix"]),
            "signup_date": f"202{random.randint(0,4)}-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
        }
    },
    "sales": {
        "columns": ["product", "price", "quantity", "date", "region"],
        "generator": lambda i: {
            "product": random.choice(["Widget A", "Widget B", "Gadget X", "Tool Y", "Part Z"]),
            "price": f"${random.randint(10, 500)}.{random.randint(0,99):02d}",
            "quantity": random.randint(1, 100),
            "date": f"{random.randint(1,12)}/{random.randint(1,28)}/202{random.randint(2,4)}",
            "region": random.choice(["North", "South", "East", "West", "Central"]),
        }
    },
    "employees": {
        "columns": ["emp_id", "full_name", "department", "salary", "hire_date"],
        "generator": lambda i: {
            "emp_id": f"EMP{i:04d}",
            "full_name": random.choice(["Michael Johnson", "Sarah Williams", "David Lee", "Emily Chen", "James Taylor"]),
            "department": random.choice(["Engineering", "Sales", "Marketing", "HR", "Finance"]),
            "salary": random.randint(40000, 150000),
            "hire_date": f"202{random.randint(0,4)}-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
        }
    },
    "inventory": {
        "columns": ["sku", "item_name", "stock", "warehouse", "last_updated"],
        "generator": lambda i: {
            "sku": f"SKU-{random.randint(1000,9999)}",
            "item_name": random.choice(["Laptop", "Monitor", "Keyboard", "Mouse", "Headphones"]),
            "stock": random.randint(0, 500),
            "warehouse": random.choice(["WH-A", "WH-B", "WH-C"]),
            "last_updated": f"202{random.randint(3,4)}-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
        }
    },
}

print("Generating datasets...")
for name, config in DATASETS.items():
    # Generate clean data
    rows = [config["generator"](i) for i in range(50)]
    clean_df = pd.DataFrame(rows)
    
    # Create messy version
    messy_df = clean_df.copy()
    for col in messy_df.columns:
        messy_df[col] = messy_df[col].apply(add_noise)
    
    # Add duplicates
    dup_rows = messy_df.sample(n=min(5, len(messy_df)))
    messy_df = pd.concat([messy_df, dup_rows], ignore_index=True)
    
    # Save
    messy_df.to_csv(tasks_dir / f"{name}_messy.csv", index=False)
    clean_df.to_csv(tasks_dir / f"{name}_clean.csv", index=False)
    print(f"  ✅ {name}: {len(messy_df)} messy → {len(clean_df)} clean")

print(f"\n✅ Generated {len(DATASETS)} dataset pairs in {tasks_dir}")

## 4. Create Training Dataset

In [ ]:
from datasets import Dataset
from pathlib import Path

# System prompt
SYSTEM_PROMPT = """You are a data cleaning expert. Clean messy CSV files using these tools:

Tools:
- read_file(path): Read the input CSV file
- run_python(code): Execute pandas code to clean data
- submit_cleaned_file(path): Submit your cleaned file

Cleaning steps:
1. read_file("input.csv") - load the data
2. run_python(code) - clean with pandas:
   - df.drop_duplicates() - remove duplicates
   - df.fillna(...) - handle missing values  
   - df['col'].str.strip() - remove whitespace
   - df.to_csv('cleaned_output.csv', index=False) - save
3. submit_cleaned_file("cleaned_output.csv") - submit

Always save to 'cleaned_output.csv' before submitting."""

# Find all messy files
tasks_dir = Path("data_clean_env/tasks")
messy_files = list(tasks_dir.glob("*_messy.csv"))

# Create prompts
prompts = []
task_descriptions = {
    "customers": "customer records with duplicates and formatting issues",
    "sales": "sales data with inconsistent prices and dates",
    "employees": "employee records with missing values",
    "inventory": "inventory data with stock discrepancies",
}

for f in messy_files:
    name = f.stem.replace("_messy", "")
    desc = task_descriptions.get(name, f"{name} data with quality issues")
    prompt = f"{SYSTEM_PROMPT}\n\nTask: Clean {desc}. File: {f.name}"
    prompts.append(prompt)

# Multiply for more training samples
prompts = prompts * 8  # ~32 samples

train_dataset = Dataset.from_dict({"prompt": prompts})
print(f"✅ Training dataset: {len(train_dataset)} samples")
print(f"   Unique tasks: {len(messy_files)}")

## 5. Define Reward Function

In [ ]:
def reward_fn(completions, **kwargs):
    """Heuristic reward function for GRPO training.
    
    Rewards proper tool usage and cleaning operations.
    No environment server needed - fast and stable.
    """
    rewards = []
    
    for completion in completions:
        if isinstance(completion, list):
            text = " ".join([str(t) for t in completion])
        else:
            text = str(completion)
        
        reward = 0.0
        
        # Step 1: Read file
        if "read_file" in text:
            reward += 0.15
        
        # Step 2: Use pandas to clean
        if "run_python" in text:
            reward += 0.20
            
            # Bonus for cleaning operations
            cleaning_ops = [
                "drop_duplicates", "fillna", "dropna", "strip",
                "to_datetime", "str.lower", "str.upper", "str.title",
                "replace", "astype", "rename", "apply"
            ]
            ops_found = sum(1 for op in cleaning_ops if op in text)
            reward += min(0.15, ops_found * 0.03)
        
        # Step 3: Save cleaned file
        if "to_csv" in text:
            reward += 0.15
            if "cleaned" in text.lower():
                reward += 0.1
        
        # Step 4: Submit
        if "submit_cleaned_file" in text:
            reward += 0.2
            if "cleaned_output.csv" in text:
                reward += 0.1
        
        # Cap at 1.0
        rewards.append(min(1.0, reward))
    
    return rewards

print("✅ Reward function defined")

## 6. Load Model with 4-bit Quantization

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

print(f"Loading model: {MODEL_ID}")

# 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

print(f"✅ Model loaded: {MODEL_ID}")
print(f"   Parameters: {model.num_parameters() / 1e9:.2f}B")
print(f"   GPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 7. Configure GRPO Trainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig

# LoRA config for efficient fine-tuning
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

# GRPO training config
training_args = GRPOConfig(
    output_dir=CONFIG["output_dir"],
    learning_rate=CONFIG["learning_rate"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    num_generations=CONFIG["num_generations"],
    max_completion_length=CONFIG["max_completion_length"],
    temperature=CONFIG["temperature"],
    logging_steps=CONFIG["logging_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    bf16=True,
    gradient_checkpointing=True,
    warmup_steps=10,
)

# Initialize trainer
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=lora_config,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ GRPO Trainer ready")
print(f"   Trainable params: {trainable / 1e6:.1f}M")
print(f"   Training samples: {len(train_dataset)}")

## 8. Train! 🚀

In [ ]:
print("=" * 50)
print("🚀 STARTING GRPO TRAINING")
print("=" * 50)
print(f"Model: {MODEL_ID}")
print(f"Samples: {len(train_dataset)}")
print(f"Epochs: {CONFIG['num_train_epochs']}")
print(f"Batch: {CONFIG['per_device_train_batch_size']} × {CONFIG['gradient_accumulation_steps']} accum")
print(f"Generations: {CONFIG['num_generations']} per prompt")
print()

# Train
train_result = trainer.train()

# Save
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print()
print("=" * 50)
print("✅ TRAINING COMPLETE!")
print("=" * 50)
print(f"Model saved to: {CONFIG['output_dir']}")
print(f"Final loss: {train_result.training_loss:.4f}")

## 9. Test the Trained Model

In [ ]:
from transformers import pipeline

# Merge LoRA weights
print("Merging LoRA weights...")
merged_model = trainer.model.merge_and_unload()

# Create pipeline
pipe = pipeline(
    "text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# Test prompt
test_prompt = f"""{SYSTEM_PROMPT}

Task: Clean customer data with duplicates and missing emails. File: customers_messy.csv"""

print("\n📝 Testing trained model...")
output = pipe(
    test_prompt,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    return_full_text=False,
)

print("\n" + "=" * 50)
print("MODEL OUTPUT:")
print("=" * 50)
print(output[0]["generated_text"])

## 10. Push to HuggingFace Hub (Optional)

In [ ]:
# Uncomment and fill in to push your model

# HF_USERNAME = "your-username"  # Your HuggingFace username
# MODEL_NAME = "dataclean-agent-grpo"

# if HF_TOKEN:
#     merged_model.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}", token=HF_TOKEN)
#     tokenizer.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}", token=HF_TOKEN)
#     print(f"✅ Pushed to https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}")
# else:
#     print("❌ Set HF_TOKEN to push model")

## 11. Download Model (to use locally)

In [ ]:
# Zip the checkpoint for download
import shutil

output_zip = f"{CONFIG['output_dir']}.zip"
shutil.make_archive(CONFIG['output_dir'], 'zip', CONFIG['output_dir'])
print(f"✅ Model zipped: {output_zip}")
print(f"   Download from Kaggle Output tab")